# Preprocessing The Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os
from pathlib import Path
import joblib

# Define the general paths of the datasets
STOCK_PATH = Path('./data/GOOGL_STOCKS/')
OCCUPANCY_PATH = Path('./data/OccupancyDetection/')
POWER_PATH = Path('./data/PowerConsumption/')
TRAFFIC_PATH = Path('./data/TrafficVolume/')

Preprocess the data by removing not relevant features and split the data into training (80%) and test (20%).


In [2]:
def preprocess_and_split_dataset(dataset_path, dataset_name, separator, decimal, type, quotechar=None):
    # 1. Construct path
    full_path = os.path.join(dataset_path, f"{dataset_name}.{type}")
    print(f"Processing: {full_path} ...")

    try:
        if quotechar:
            df = pd.read_csv(full_path, sep=separator, decimal=decimal, quotechar=quotechar)
        else:
            df = pd.read_csv(full_path, sep=separator, decimal=decimal)
    except FileNotFoundError:
        print(f"Error: File not found at {full_path}")
        return

    df.replace("None", pd.NA, inplace=True)
    df.dropna(axis=1, how='all', inplace=True)
    df.dropna(axis=0, how='all', inplace=True)
    drop_cols = ['holiday', 'weather_description']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    rename_cols = ['date', 'DateTime', 'date_time']
    df = df.rename(columns={c: "Date" for c in rename_cols if c in df.columns})


    split_idx = int(len(df) * 0.80)
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    train_path = os.path.join(dataset_path, f"{dataset_name}_train.csv")
    test_path = os.path.join(dataset_path, f"{dataset_name}_test.csv")
    
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)
    
    print(f"Saved Train: {len(train_df)} rows | Test: {len(test_df)} rows")

preprocess_and_split_dataset('./data/GOOGL_STOCKS', 'GOOGL_STOCKS', separator=',', decimal='.', type='csv')
preprocess_and_split_dataset('./data/OccupancyDetection', 'OccupancyDetection', separator=',', decimal='.', type='txt', quotechar='"') 
preprocess_and_split_dataset('./data/PowerConsumption', 'PowerConsumption', separator=',', decimal='.', type='csv')
preprocess_and_split_dataset('./data/TrafficVolume', 'TrafficVolume', separator=',', decimal='.', type='csv')

Processing: ./data/GOOGL_STOCKS\GOOGL_STOCKS.csv ...
Saved Train: 3544 rows | Test: 887 rows
Processing: ./data/OccupancyDetection\OccupancyDetection.txt ...
Saved Train: 6514 rows | Test: 1629 rows
Processing: ./data/PowerConsumption\PowerConsumption.csv ...
Saved Train: 5184 rows | Test: 1297 rows
Processing: ./data/TrafficVolume\TrafficVolume.csv ...
Saved Train: 4823 rows | Test: 1206 rows


Define a Dataset configuration class to handle easily different datasets

In [3]:
class DatasetConfig:
    def __init__(self, name, csv_path, sep, decimal):
        """
        dataset_type: 'cyclical' (Air, Energy, Beijing) or 'financial' (Stocks)
        """
        self.name = name
        self.csv_path = csv_path
        self.sep = sep
        self.decimal = decimal

Define the Data processor to process the data for different models.

In [4]:
class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.scaler = MinMaxScaler()
        self.raw_df = None
        self.processed_df = None  # For GANs/Chronos
        self.text_data = None     # For LLM

    def load_and_clean(self):
        df = pd.read_csv(self.config.csv_path, sep=self.config.sep, decimal=self.config.decimal)
        
        if 'Date' in df.columns:
             df['Date_Combined'] = pd.to_datetime(df['Date'])
        
        df = df.sort_values('Date_Combined')

        # 3. Fill NAs
        df = df.ffill().bfill()
        self.raw_df = df
        return self

    def prepare_for_numerical_models(self):
        df = self.raw_df.copy()
        
        if "weather_main" in df.columns:
            df.drop(columns=["weather_main"])
        df_numeric = df.select_dtypes(include=[np.number])
        scaled_values = self.scaler.fit_transform(df_numeric)
        processed_df = pd.DataFrame(scaled_values, columns=df_numeric.columns)

        self.processed_df = processed_df
        return self.processed_df


    def prepare_for_llm(self):
        df = self.raw_df.copy()
        
        if 'Date_Combined' in df.columns:
            df['Date'] = df['Date_Combined'].dt.strftime('%Y-%m-%d %H:%M')
            df = df.drop(columns=['Date_Combined'])

        df = df.round(2)
        
        self.text_data = df.to_dict(orient='records')
        
        return self.text_data

    def get_data(self):
        return {
            "numerical": self.processed_df,
            "text": self.text_data,
            "scaler": self.scaler
        }

Process the different versions of data for the models.

In [5]:
configs = [
    DatasetConfig("GOOGL_STOCKS", STOCK_PATH / 'GOOGL_STOCKS_train.csv', sep=',', decimal='.'),
    DatasetConfig("OccupancyDetection", OCCUPANCY_PATH / 'OccupancyDetection_train.csv', sep=',', decimal='.'),
    DatasetConfig("PowerConsumption", POWER_PATH / 'PowerConsumption_train.csv', sep=',', decimal='.'),
    DatasetConfig("TrafficVolume", TRAFFIC_PATH / 'TrafficVolume_train.csv', sep=',', decimal='.')
]

processed_datasets = {}

for cfg in configs:
    processor = DataProcessor(cfg)
    # Run Pipeline
    processor.load_and_clean()
    processor.prepare_for_numerical_models() 
    processor.prepare_for_llm()              
    # Store results
    processed_datasets[cfg.name] = processor.get_data()

Save the data and scaler.

In [6]:
def save_processed_data(processed_datasets, dataset_dir, scaler_dir):
    
    os.makedirs(scaler_dir, exist_ok=True)
    for name, data in processed_datasets.items():
        save_dir = os.path.join(dataset_dir, name)
        os.makedirs(save_dir, exist_ok=True)
        os.makedirs(scaler_dir, exist_ok=True)

        num_path = os.path.join(save_dir, f"{name}_numerical.csv")
        data['numerical'].to_csv(num_path, index=False)

        text_path = os.path.join(save_dir, f"{name}_llm_text.csv")
        pd.DataFrame(data['text']).to_csv(text_path, index=False)

        scaler_path = os.path.join(scaler_dir, f"{name}_scaler.pkl")
        joblib.dump(data['scaler'], scaler_path)

        
save_processed_data(processed_datasets, './data', './scaler')